# PySpark: Gold Layer Consolidado
Pipeline PySpark que une os 3 datasets da camada Prata e gera a camada Ouro.

**Pré-requisito:** Executar TODAS as células (incluindo a seção PySpark) dos 3 notebooks individuais (`incedets_master.ipynb`, `financial_impact.ipynb`, `market_impact.ipynb`) para gerar os arquivos Silver Spark (`*_spark.parquet`).

### Etapas:
1. Window Functions: enriquece cada Silver com métricas analíticas
2. Gold Layer: join dos 3 datasets + agregações + rankings
3. Comparação de desempenho Pandas vs PySpark
4. Tabela de equivalência de operações

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    current_timestamp, lit, col, when, trim, lower,
    to_timestamp, avg, rank, desc, count, date_format
)
from pyspark.sql.window import Window
import time, numpy as np, pandas as pd, subprocess, sys

# === Verifica se Java esta instalado ===
java_ok = False
try:
    subprocess.run(["java", "-version"], capture_output=True, timeout=5)
    java_ok = True
except:
    java_ok = False

if not java_ok:
    print("=" * 60)
    print("ERRO: Java nao encontrado! PySpark exige Java JDK 11+.")
    print("Instale o Java (https://adoptium.net) e configure JAVA_HOME.")
    print("=" * 60)
    spark = None
else:
    spark = SparkSession.builder \
        .appName("GoldLayerPySpark") \
        .config("spark.sql.adaptive.enabled", "true") \
        .getOrCreate()
    print("SparkSession OK")

---
## 1. Window Functions
Operações analíticas nativas do Spark aplicadas sobre cada dataset Silver.

### 1.1 Window - incedets_master
Média móvel de downtime por tipo de ataque + ranking de downtime por setor.

In [ ]:
if spark:
    t0 = time.time()

    df_m = spark.read.parquet("Dados/prata/incedets_master_silver_spark.parquet")

    df_w = df_m.withColumn(
        "avg_downtime_por_ataque",
        avg("downtime_hours").over(
            Window.partitionBy("attack_vector_primary")
                  .orderBy("incident_date")
                  .rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    ).withColumn(
        "rank_downtime_setor",
        rank().over(
            Window.partitionBy("industry_primary")
                  .orderBy(desc("downtime_hours"))
        )
    )

    df_w.write.mode("overwrite").parquet("Dados/ouro/window_incedets_master.parquet")
    t_window = time.time() - t0
    print(f"Window incedets_master: {t_window:.4f}s  Colunas adicionadas")
else:
    print("Spark nao disponivel. Pule esta celula.")

### 1.2 Window - financial_impact
Média móvel de perda total por método de cálculo + ranking de perda por índice CPI.

In [ ]:
if spark:
    t0 = time.time()

    df_f = spark.read.parquet("Dados/prata/financial_impact_prata_spark.parquet")

    df_w = df_f.withColumn(
        "avg_total_loss_por_metodo",
        avg("total_loss_usd").over(
            Window.partitionBy("direct_loss_method")
                  .orderBy("created_at")
                  .rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    ).withColumn(
        "rank_perda_por_cpi",
        rank().over(
            Window.partitionBy("cpi_index_used")
                  .orderBy(desc("total_loss_usd"))
        )
    )

    df_w.write.mode("overwrite").parquet("Dados/ouro/window_financial_impact.parquet")
    t_window = time.time() - t0
    print(f"Window financial_impact: {t_window:.4f}s  Colunas adicionadas")
else:
    print("Spark nao disponivel. Pule esta celula.")

### 1.3 Window - market_impact
Média móvel de market cap por setor + ranking de abnormal return por ticker.

In [ ]:
if spark:
    t0 = time.time()

    df_k = spark.read.parquet("Dados/prata/market_silver_completo_spark.parquet")

    df_w = df_k.withColumn(
        "avg_marketcap_por_setor",
        avg("market_cap_at_disclosure").over(
            Window.partitionBy("sector_index")
                  .orderBy("created_at")
                  .rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    ).withColumn(
        "rank_abnormal_return",
        rank().over(
            Window.partitionBy("stock_ticker")
                  .orderBy(desc("abnormal_return_30d"))
        )
    )

    df_w.write.mode("overwrite").parquet("Dados/ouro/window_market_impact.parquet")
    t_window = time.time() - t0
    print(f"Window market_impact: {t_window:.4f}s  Colunas adicionadas")
else:
    print("Spark nao disponivel. Pule esta celula.")

---
## 2. Gold Layer - Join + Agregações + Rankings
Junção dos 3 datasets Silver (via `incident_id`) e geração da camada Ouro.

In [ ]:
if spark:
    t0 = time.time()

    m = spark.read.parquet("Dados/prata/incedets_master_silver_spark.parquet")
    f = spark.read.parquet("Dados/prata/financial_impact_prata_spark.parquet")
    k = spark.read.parquet("Dados/prata/market_silver_completo_spark.parquet")

    # Remove colunas duplicadas antes de cada join (evita COLUMN_ALREADY_EXISTS)
    dup_mf = [c for c in f.columns if c != "incident_id" and c in m.columns]
    f_clean = f.drop(*dup_mf)

    gold = m.join(f_clean, on="incident_id", how="left")

    # Segundo join: remove colunas de k que ja existem no resultado anterior
    dup_mk = [c for c in k.columns if c != "incident_id" and c in gold.columns]
    k_clean = k.drop(*dup_mk)
    gold = gold.join(k_clean, on="incident_id", how="left")

    print(f"Gold: {gold.count()} registros, {len(gold.columns)} colunas")

    # --- groupBy().agg() (antes: df.groupby().agg()) ---
    agg_setor = gold.groupBy("industry_primary").agg(
        avg("downtime_hours").alias("avg_downtime_h"),
        avg("total_loss_usd").alias("avg_total_loss"),
        avg("market_cap_at_disclosure").alias("avg_marketcap"),
        count("*").alias("total_incidentes")
    )

    # Window sobre agregacao: ranking setores por prejuizo medio
    agg_setor = agg_setor.withColumn(
        "rank_setor_prejuizo",
        rank().over(Window.orderBy(desc("avg_total_loss")))
    )

    agg_setor.show(10, truncate=False)

    gold.write.mode("overwrite").parquet("Dados/ouro/gold_completo.parquet")
    agg_setor.write.mode("overwrite").parquet("Dados/ouro/gold_agregacoes_setor.parquet")
    t_gold = time.time() - t0
    print(f"Gold criada: {t_gold:.4f}s")
else:
    print("Spark nao disponivel. Pule esta celula.")

---
## 3. Comparação Pandas vs PySpark
Benchmark de desempenho entre as duas implementações para cada dataset.

### 3.1 incedets_master

In [ ]:
if spark:
    # --- PANDAS ---
    t0 = time.time()
    df_pd = pd.read_csv("Dados/originais/incidents_master.csv")
    df_pd = df_pd.copy()
    df_pd = df_pd.drop_duplicates(subset=['incident_id'], keep='last')
    for col_pd in ["company_revenue_usd", "employee_count", "data_compromised_records", "downtime_hours"]:
        if col_pd in df_pd.columns:
            df_pd[col_pd] = df_pd[col_pd].fillna(0)
    t_pd = time.time() - t0

    # --- PySpark ---
    t0 = time.time()
    df_sp = spark.read.parquet("Dados/bronze/incedets_master_bronze_spark.parquet")
    df_sp = df_sp.dropDuplicates(["incident_id"])
    for col_sp in ["company_revenue_usd", "employee_count", "data_compromised_records", "downtime_hours"]:
        df_sp = df_sp.fillna(0, subset=[col_sp])
    count_sp = df_sp.count()
    t_sp = time.time() - t0

    print('=' * 60)
    print('COMPARACAO incedets_master: PANDAS vs PySpark')
    print('=' * 60)
    print(f'  Pandas:  {t_pd:.4f}s')
    print(f'  PySpark: {t_sp:.4f}s')
    ratio = t_pd / t_sp if t_sp > 0 else 0
    print(f'  Speedup: {ratio:.2f}x')
    print('=' * 60)
else:
    print("Spark nao disponivel. Pule esta celula.")

### 3.2 financial_impact

In [ ]:
if spark:
    # --- PANDAS ---
    t0 = time.time()
    df_pd = pd.read_csv("Dados/originais/financial_impact.csv")
    df_pd = df_pd.copy()
    df_pd = df_pd.drop_duplicates(subset=['incident_id'], keep='last')
    for col_pd in ["direct_loss_usd", "ransom_demanded_usd", "ransom_paid_usd", "recovery_cost_usd", "legal_fees_usd", "regulatory_fine_usd", "insurance_payout_usd", "total_loss_usd", "total_loss_lower_bound", "total_loss_upper_bound", "inflation_adjusted_usd"]:
        if col_pd in df_pd.columns:
            df_pd[col_pd] = df_pd[col_pd].fillna(0)
    t_pd = time.time() - t0

    # --- PySpark ---
    t0 = time.time()
    df_sp = spark.read.parquet("Dados/bronze/financial_impact_bronze_spark.parquet")
    df_sp = df_sp.dropDuplicates(["incident_id"])
    for col_sp in ["direct_loss_usd", "ransom_demanded_usd", "ransom_paid_usd", "recovery_cost_usd", "legal_fees_usd", "regulatory_fine_usd", "insurance_payout_usd", "total_loss_usd", "total_loss_lower_bound", "total_loss_upper_bound", "inflation_adjusted_usd"]:
        df_sp = df_sp.fillna(0, subset=[col_sp])
    count_sp = df_sp.count()
    t_sp = time.time() - t0

    print('=' * 60)
    print('COMPARACAO financial_impact: PANDAS vs PySpark')
    print('=' * 60)
    print(f'  Pandas:  {t_pd:.4f}s')
    print(f'  PySpark: {t_sp:.4f}s')
    ratio = t_pd / t_sp if t_sp > 0 else 0
    print(f'  Speedup: {ratio:.2f}x')
    print('=' * 60)
else:
    print("Spark nao disponivel. Pule esta celula.")

### 3.3 market_impact

In [ ]:
if spark:
    # --- PANDAS ---
    t0 = time.time()
    df_pd = pd.read_csv("Dados/originais/market_impact.csv")
    df_pd = df_pd.copy()
    df_pd = df_pd.drop_duplicates(subset=['incident_id'], keep='last')
    for col_pd in ["market_cap_at_disclosure", "volume_ratio_disclosure", "pre_incident_volatility_30d", "post_incident_volatility_30d", "days_to_price_recovery", "car_0_to_30", "abnormal_return_30d", "t_statistic_30d", "p_value_30d"]:
        if col_pd in df_pd.columns:
            df_pd[col_pd] = df_pd[col_pd].fillna(0)
    t_pd = time.time() - t0

    # --- PySpark ---
    t0 = time.time()
    df_sp = spark.read.parquet("Dados/bronze/market_impact_bronze_spark.parquet")
    df_sp = df_sp.dropDuplicates(["incident_id"])
    for col_sp in ["market_cap_at_disclosure", "volume_ratio_disclosure", "pre_incident_volatility_30d", "post_incident_volatility_30d", "days_to_price_recovery", "car_0_to_30", "abnormal_return_30d", "t_statistic_30d", "p_value_30d"]:
        df_sp = df_sp.fillna(0, subset=[col_sp])
    count_sp = df_sp.count()
    t_sp = time.time() - t0

    print('=' * 60)
    print('COMPARACAO market_impact: PANDAS vs PySpark')
    print('=' * 60)
    print(f'  Pandas:  {t_pd:.4f}s')
    print(f'  PySpark: {t_sp:.4f}s')
    ratio = t_pd / t_sp if t_sp > 0 else 0
    print(f'  Speedup: {ratio:.2f}x')
    print('=' * 60)
else:
    print("Spark nao disponivel. Pule esta celula.")

---
## 4. Tabela de Equivalência Pandas -> PySpark

| Operação | Pandas | PySpark |
|---|---|---|
| Leitura | `pd.read_csv()` | `spark.read.csv()` |
| Auditoria | `df['col'] = dt.now()` | `.withColumn(..., current_timestamp())` |
| Deduplicação | `df.drop_duplicates()` | `df.dropDuplicates()` |
| Fill nulos | `df[col].fillna(0)` | `df.fillna(0, subset=[col])` |
| String | `str.lower().strip()` | `lower(trim(col()))` |
| Datas | `pd.to_datetime()` | `to_timestamp(col())` |
| Label | `np.where(cond, 1, 0)` | `when(col>0,1).otherwise(0)` |
| Anti-leakage | `df.drop(columns=...)` | `df.drop(*cols)` |
| Join | `pd.merge(df1, df2, on=key)` | `df1.join(df2, on=key, how=...)` |
| Agregação | `df.groupby('c').agg(...)` | `df.groupBy('c').agg(...)` |
| Window | Não nativa | `Window.partitionBy().orderBy()` |
| Salvar | `df.to_parquet()` | `df.write.parquet()` |

---
### Conclusão

A camada Ouro foi gerada com sucesso pelo PySpark:
- `Dados/ouro/gold_completo.parquet` — Join completo dos 3 datasets
- `Dados/ouro/gold_agregacoes_setor.parquet` — Agregações por setor com ranking
- `Dados/ouro/window_incedets_master.parquet` — Window: média móvel + ranking
- `Dados/ouro/window_financial_impact.parquet` — Window: média móvel + ranking
- `Dados/ouro/window_market_impact.parquet` — Window: média móvel + ranking

O PySpark demonstra vantagens significativas em escalabilidade horizontal (mais workers = mais velocidade), enquanto o pandas se mantém superior para datasets pequenos (< 1M registros) pela simplicidade e ecossistema de visualização.